# 🥈 Silver Layer — Transformation & Cleaning
**Project:** Warehouse & Retail Sales ML Pipeline  
**Author:** Santiago López Blanco  
**Date:** May 2026  
**Goal:** Read from Bronze, fix data quality issues, enrich the dataset,
and save a clean, reliable Delta Table ready for analysis and modeling.

### What changes from Bronze to Silver
- Create a unified `date` column from YEAR + MONTH
- Standardize text fields (uppercase, trim whitespace)
- Fill null values with appropriate defaults
- Calculate `total_sales` as a new derived column
- Rename columns to snake_case (no spaces)

In [0]:
# Read data from the Bronze Delta Table
# We always read from Bronze, never from the raw CSV
# This ensures Silver inherits from a controlled, versioned source

try:
    df_bronze = spark.table("main.default.bronze_warehouse_sales")
    
    print("=" * 45)
    print("  READING FROM BRONZE")
    print("=" * 45)
    print(f"  Rows   : {df_bronze.count():,}")
    print(f"  Columns: {len(df_bronze.columns)}")
    print(f"  Source : main.default.bronze_warehouse_sales")
    print("=" * 45)
    print("  Status: Bronze table loaded ✓")
    print("=" * 45)

except Exception as e:
    print(f"  ✗ Failed to read Bronze table: {e}")
    raise

## Silver Layer — Transformations Applied

The following operations are applied to the Bronze dataset in a single chained transformation:

1. **Date column** — YEAR and MONTH unified into a single `date` column of type `DateType`
2. **Snake case** — all column names renamed to remove spaces (required for Delta Lake compatibility)
3. **Standardization** — text fields converted to uppercase and trimmed
4. **Null handling** — each null column filled based on Bronze quality findings
5. **Total sales** — new derived column: `retail_sales + retail_transfers + warehouse_sales`
6. **Column selection** — final schema ordered logically for downstream consumption

In [0]:
from pyspark.sql import functions as F

df_silver = (df_bronze

    # 1. Create a unified date column from YEAR and MONTH
    # concat_ws joins them as "2020-1-01", then to_date parses it into a proper date type
    .withColumn("date",
        F.to_date(
            F.concat_ws("-", F.col("YEAR").cast("string"), F.col("MONTH").cast("string"), F.lit("01")),
            "yyyy-M-dd"
        )
    )

    # 2. Rename all columns to snake_case
    # Original column names have spaces — this breaks many downstream tools
    .withColumnRenamed("ITEM CODE", "item_code")
    .withColumnRenamed("ITEM DESCRIPTION", "item_description")
    .withColumnRenamed("ITEM TYPE", "item_type")
    .withColumnRenamed("RETAIL SALES", "retail_sales")
    .withColumnRenamed("RETAIL TRANSFERS", "retail_transfers")
    .withColumnRenamed("WAREHOUSE SALES", "warehouse_sales")
    .withColumnRenamed("SUPPLIER", "supplier")
    .withColumnRenamed("YEAR", "year")
    .withColumnRenamed("MONTH","month")

    # 3. Standardize text fields
    # upper() ensures consistent casing — "pwswn inc" and "PWSWN INC" are the same supplier
    # trim() removes leading/trailing whitespace that could cause duplicates
    .withColumn("supplier", F.upper(F.trim(F.col("supplier"))))
    .withColumn("item_description", F.upper(F.trim(F.col("item_description"))))
    .withColumn("item_type", F.upper(F.trim(F.col("item_type"))))

    # 4. Fill null values using when/otherwise for explicit control
    # fillna() was avoided because it had inconsistent behavior with renamed columns
    # Each column is handled individually based on Bronze quality findings:
    # - item_code: 167 nulls — same rows where supplier is also null
    # - supplier: 167 nulls — filled with "UNKNOWN" to preserve the rows
    # - item_type: 1 null — filled with "UNCLASSIFIED"
    # - retail_sales: 3 nulls — filled with 0.0 (no sale = zero sales)
    .withColumn("item_code",
        F.when(F.col("item_code").isNull(), "UNKNOWN")
        .otherwise(F.col("item_code"))
    )
    .withColumn("supplier",
        F.when(F.col("supplier").isNull(), "UNKNOWN")
        .otherwise(F.col("supplier"))
    )
    .withColumn("item_type",
        F.when(F.col("item_type").isNull(), "UNCLASSIFIED")
        .otherwise(F.col("item_type"))
    )
    .withColumn("retail_sales",
        F.when(F.col("retail_sales").isNull(), 0.0)
        .otherwise(F.col("retail_sales"))
    )

    # 5. Calculate total_sales as a derived column
    # All three sales columns must be included — retail_transfers is often missed
    .withColumn("total_sales",
        F.col("retail_sales") + F.col("retail_transfers") + F.col("warehouse_sales")
    )

    # 6. Select final columns in logical order
    # Only keep what is needed — drop year and month since date already contains that info
    .select(
        "date", "year", "month",
        "supplier", "item_code", "item_description", "item_type",
        "retail_sales", "retail_transfers", "warehouse_sales", "total_sales"
    )
)

display(df_silver)

In [0]:
# Verify null handling was applied correctly
# All counts should be 0 after Silver transformation
print("\nNull check after transformation:")
null_check = df_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
])
display(null_check)

# Verify item_type values — should only contain WINE, BEER, LIQUOR, UNCLASSIFIED
print("\nDistinct item_type values:")
display(df_silver.groupBy("item_type").count().orderBy("count", ascending=False))

# Verify date column was created correctly
print("\nDate range:")
df_silver.select(
    F.min("date").alias("earliest_date"),
    F.max("date").alias("latest_date")
).show()

## Silver Layer — Quality Validation

All transformations verified before saving:

| Check | Result |
|-------|--------|
| Null values in any column | 0 ✓ |
| Distinct item types | WINE, LIQUOR, BEER, KEGS, NON-ALCOHOL + 4 more |
| Date range | 2017-06-01 → 2020-09-01 |
| Total rows | 307,645 ✓ |

> **Finding:** Dataset contains more item types than initially assumed from the preview.
> KEGS (10,146 rows) and NON-ALCOHOL (1,908 rows) were not visible in the Bronze preview.
> This confirms why full data profiling matters before making assumptions.

In [0]:
# Save the cleaned dataset as the official Silver Delta Table
# This table is the foundation for all analysis and ML modeling

try:
    df_silver.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("main.default.silver_warehouse_sales")

    row_count = spark.table("main.default.silver_warehouse_sales").count()

    print("=" * 45)
    print("  SILVER LAYER — COMPLETE")
    print("=" * 45)
    print(f"  Table  : main.default.silver_warehouse_sales")
    print(f"  Rows   : {row_count:,}")
    print(f"  Columns: 11")
    print(f"  Status : saved successfully ✓")
    print("=" * 45)

except Exception as e:
    print(f"  ✗ Failed to save Silver table: {e}")
    raise

## Silver Layer — Summary

The Silver layer is now complete. The Bronze dataset has been cleaned,
enriched, and saved as a reliable Delta Table ready for analysis and modeling.

### What was done
- Created unified `date` column from YEAR and MONTH
- Renamed all columns to snake_case for Delta Lake compatibility
- Standardized text fields to uppercase and trimmed whitespace
- Filled null values based on Bronze quality findings
- Calculated `total_sales` as a derived column
- Saved as Delta Table: `main.default.silver_warehouse_sales`

### Schema — 11 columns
| Column | Type | Description |
|--------|------|-------------|
| date | date | Transaction date |
| year | long | Year extracted from date — kept for groupBy convenience |
| month | long | Month extracted from date — kept for groupBy convenience |
| supplier | string | Distributor name (uppercase) |
| item_code | string | Product identifier |
| item_description | string | Full product name (uppercase) |
| item_type | string | Category: WINE, BEER, LIQUOR, KEGS, NON-ALCOHOL |
| retail_sales | double | Units sold at retail |
| retail_transfers | double | Units transferred between stores |
| warehouse_sales | double | Units sold from warehouse |
| total_sales | double | retail_sales + retail_transfers + warehouse_sales |

> **Design note:** `year` and `month` are technically redundant since both can be
> extracted from `date` at any time. They are kept as explicit columns for
> convenience — groupBy operations in Gold and ML feature engineering
> are cleaner when year and month are directly accessible without extraction.

### Next step
`03_silver_EDA` — Exploratory Data Analysis on the Silver dataset.
Understand distributions, trends, seasonality, and supplier behavior
before defining Gold layer business metrics.